In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-14B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


## Qwen-14B-Instruct 

## Qwen-14B-Instruct on Llama 70B

In [3]:
pip install torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-14B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████| 8/8 [00:05<00:00,  1.42it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [3]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "14B_High_Low_Irr_Data_Source_.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None

# Create results dataframe
results = []

# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["High"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        query_full = (
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Based on the information provided, select the correct answer choice (A, B, C, D, etc.).\n\n"
            "IMPORTANT: Your response must end with 'Answer: X' where X is the letter of your chosen option.\n"
            "For example: Answer: A"
    )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
        )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
            qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(paths.PREDICTIONS / "Qwen_14B_predictions_progress.csv", index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "Qwen_14B_predictions_14B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")


Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options', 'answer_corr']
Processing 1300 rows...
Processing row 1/1300...
✅ Processed Merge Q9: Answer = D
Processing row 2/1300...
✅ Processed Merge Q9: Answer = D
Processing row 3/1300...
✅ Processed Merge Q9: Answer = A
Processing row 4/1300...
✅ Processed Merge Q9: Answer = D
Processing row 5/1300...
✅ Processed Merge Q9: Answer = H
Processing row 6/1300...
✅ Processed Merge Q9: Answer = B
Processing row 7/1300...
✅ Processed Merge Q9: Answer = C
Processing row 8/1300...
✅ Processed Merge Q9: Answer = B
Processing row 9/1300...
✅ Processed Merge Q9: Answer = C
Processing row 10/1300...
✅ Processed Merge Q9: Answer = D
Saved progress to CSV after 10 items
Processing row 11/1300...
✅ Processed Merge Q9: Answer = A
Processing row 12/1300...
✅ Processed Merge Q9: Answer = B
Processing row 13/1300...
✅ Processed Merge Q9: Answer = B
Processing row 14/1300...
✅ Processed Merge Q9: Answer = D

✅ Processed Merge Q62: Answer = B
Processing row 124/1300...
⚠️ Could not extract answer from response for row 124:
Response: To determine the most likely diagnosis for this patient, we need to consider her symptoms and physic...
✅ Processed Merge Q124: Answer = None
Processing row 125/1300...
✅ Processed Merge Q124: Answer = D
Processing row 126/1300...
✅ Processed Merge Q124: Answer = D
Processing row 127/1300...
✅ Processed Merge Q124: Answer = G
Processing row 128/1300...
✅ Processed Merge Q124: Answer = D
Processing row 129/1300...
✅ Processed Merge Q124: Answer = F
Processing row 130/1300...
✅ Processed Merge Q124: Answer = C
Saved progress to CSV after 130 items
Processing row 131/1300...
✅ Processed Merge Q124: Answer = D
Processing row 132/1300...
✅ Processed Merge Q124: Answer = I
Processing row 133/1300...
✅ Processed Merge Q124: Answer = C
Processing row 134/1300...
✅ Processed Merge Q124: Answer = B
Processing row 135/1300...
✅ Processed Merge Q124: Answer = B
Processing r

✅ Processed Merge Q185: Answer = F
Processing row 242/1300...
✅ Processed Merge Q185: Answer = C
Processing row 243/1300...
✅ Processed Merge Q185: Answer = B
Processing row 244/1300...
✅ Processed Merge Q185: Answer = B
Processing row 245/1300...
✅ Processed Merge Q185: Answer = B
Processing row 246/1300...
✅ Processed Merge Q185: Answer = A
Processing row 247/1300...
✅ Processed Merge Q185: Answer = B
Processing row 248/1300...
✅ Processed Merge Q185: Answer = D
Processing row 249/1300...
✅ Processed Merge Q185: Answer = A
Processing row 250/1300...
✅ Processed Merge Q185: Answer = D
Saved progress to CSV after 250 items
Processing row 251/1300...
✅ Processed Merge Q185: Answer = D
Processing row 252/1300...
✅ Processed Merge Q185: Answer = B
Processing row 253/1300...
✅ Processed Merge Q185: Answer = F
Processing row 254/1300...
✅ Processed Merge Q185: Answer = D
Processing row 255/1300...
✅ Processed Merge Q185: Answer = B
Processing row 256/1300...
✅ Processed Merge Q185: Answer =

✅ Processed Merge Q185: Answer = H
Processing row 367/1300...
✅ Processed Merge Q185: Answer = B
Processing row 368/1300...
✅ Processed Merge Q185: Answer = D
Processing row 369/1300...
✅ Processed Merge Q185: Answer = C
Processing row 370/1300...
✅ Processed Merge Q185: Answer = C
Saved progress to CSV after 370 items
Processing row 371/1300...
✅ Processed Merge Q185: Answer = D
Processing row 372/1300...
✅ Processed Merge Q185: Answer = C
Processing row 373/1300...
✅ Processed Merge Q185: Answer = D
Processing row 374/1300...
✅ Processed Merge Q185: Answer = C
Processing row 375/1300...
✅ Processed Merge Q185: Answer = E
Processing row 376/1300...
✅ Processed Merge Q185: Answer = A
Processing row 377/1300...
✅ Processed Merge Q185: Answer = D
Processing row 378/1300...
✅ Processed Merge Q185: Answer = C
Processing row 379/1300...
✅ Processed Merge Q185: Answer = D
Processing row 380/1300...
✅ Processed Merge Q185: Answer = D
Saved progress to CSV after 380 items
Processing row 381/13

✅ Processed Merge Q185: Answer = C
Processing row 492/1300...
✅ Processed Merge Q185: Answer = C
Processing row 493/1300...
✅ Processed Merge Q185: Answer = C
Processing row 494/1300...
✅ Processed Merge Q185: Answer = C
Processing row 495/1300...
✅ Processed Merge Q185: Answer = A
Processing row 496/1300...
✅ Processed Merge Q185: Answer = D
Processing row 497/1300...
✅ Processed Merge Q185: Answer = A
Processing row 498/1300...
✅ Processed Merge Q185: Answer = C
Processing row 499/1300...
✅ Processed Merge Q185: Answer = J
Processing row 500/1300...
✅ Processed Merge Q185: Answer = C
Saved progress to CSV after 500 items
Processing row 501/1300...
✅ Processed Merge Q185: Answer = C
Processing row 502/1300...
✅ Processed Merge Q185: Answer = C
Processing row 503/1300...
✅ Processed Merge Q185: Answer = D
Processing row 504/1300...
✅ Processed Merge Q185: Answer = B
Processing row 505/1300...
✅ Processed Merge Q185: Answer = B
Processing row 506/1300...
✅ Processed Merge Q185: Answer =

...

✅ Processed Merge Q721: Answer = C
Processing row 735/1300...
✅ Processed Merge Q721: Answer = D
Processing row 736/1300...
✅ Processed Merge Q721: Answer = B
Processing row 737/1300...
✅ Processed Merge Q721: Answer = A
Processing row 738/1300...
✅ Processed Merge Q721: Answer = B
Processing row 739/1300...
✅ Processed Merge Q721: Answer = C
Processing row 740/1300...
✅ Processed Merge Q721: Answer = D
Saved progress to CSV after 740 items
Processing row 741/1300...
✅ Processed Merge Q721: Answer = G
Processing row 742/1300...
✅ Processed Merge Q721: Answer = D
Processing row 743/1300...
✅ Processed Merge Q721: Answer = A
Processing row 744/1300...
✅ Processed Merge Q721: Answer = C
Processing row 745/1300...
✅ Processed Merge Q721: Answer = A
Processing row 746/1300...
✅ Processed Merge Q721: Answer = C
Processing row 747/1300...
✅ Processed Merge Q721: Answer = D
Processing row 748/1300...
✅ Processed Merge Q721: Answer = D
Processing row 749/1300...
✅ Processed Merge Q721: Answer =

⚠️ Could not extract answer from response for row 851:
Response: To determine which medication operates through the same cellular signaling pathway as the one activa...
✅ Processed Merge Q851: Answer = None
Processing row 852/1300...
⚠️ Could not extract answer from response for row 852:
Response: To determine the appropriate antibiotic regimen for this patient, we need to consider several factor...
✅ Processed Merge Q852: Answer = None
Processing row 853/1300...
✅ Processed Merge Q852: Answer = D
Processing row 854/1300...
✅ Processed Merge Q852: Answer = B
Processing row 855/1300...
✅ Processed Merge Q852: Answer = B
Processing row 856/1300...
✅ Processed Merge Q852: Answer = B
Processing row 857/1300...
✅ Processed Merge Q852: Answer = B
Processing row 858/1300...
✅ Processed Merge Q852: Answer = C
Processing row 859/1300...
✅ Processed Merge Q852: Answer = D
Processing row 860/1300...
✅ Processed Merge Q852: Answer = C
Saved progress to CSV after 860 items
Processing row 861/1300..

✅ Processed Merge Q935: Answer = B
Processing row 967/1300...
✅ Processed Merge Q935: Answer = B
Processing row 968/1300...
✅ Processed Merge Q935: Answer = A
Processing row 969/1300...
✅ Processed Merge Q935: Answer = C
Processing row 970/1300...
✅ Processed Merge Q935: Answer = E
Saved progress to CSV after 970 items
Processing row 971/1300...
✅ Processed Merge Q935: Answer = D
Processing row 972/1300...
✅ Processed Merge Q935: Answer = D
Processing row 973/1300...
✅ Processed Merge Q935: Answer = B
Processing row 974/1300...
✅ Processed Merge Q935: Answer = C
Processing row 975/1300...
✅ Processed Merge Q935: Answer = D
Processing row 976/1300...
✅ Processed Merge Q935: Answer = B
Processing row 977/1300...
✅ Processed Merge Q935: Answer = C
Processing row 978/1300...
✅ Processed Merge Q935: Answer = A
Processing row 979/1300...
✅ Processed Merge Q935: Answer = A
Processing row 980/1300...
✅ Processed Merge Q935: Answer = B
Saved progress to CSV after 980 items
Processing row 981/13

✅ Processed Merge Q935: Answer = C
Saved progress to CSV after 1090 items
Processing row 1091/1300...
✅ Processed Merge Q935: Answer = I
Processing row 1092/1300...
✅ Processed Merge Q935: Answer = A
Processing row 1093/1300...
✅ Processed Merge Q935: Answer = F
Processing row 1094/1300...
✅ Processed Merge Q935: Answer = A
Processing row 1095/1300...
✅ Processed Merge Q935: Answer = J
Processing row 1096/1300...
✅ Processed Merge Q935: Answer = D
Processing row 1097/1300...
✅ Processed Merge Q935: Answer = C
Processing row 1098/1300...
✅ Processed Merge Q935: Answer = B
Processing row 1099/1300...
✅ Processed Merge Q935: Answer = C
Processing row 1100/1300...
✅ Processed Merge Q935: Answer = C
Saved progress to CSV after 1100 items
Processing row 1101/1300...
✅ Processed Merge Q935: Answer = C
Processing row 1102/1300...
✅ Processed Merge Q935: Answer = B
Processing row 1103/1300...
✅ Processed Merge Q935: Answer = B
Processing row 1104/1300...
✅ Processed Merge Q935: Answer = B
Proce

✅ Processed Merge Q1171: Answer = A
Processing row 1207/1300...
✅ Processed Merge Q1171: Answer = A
Processing row 1208/1300...
✅ Processed Merge Q1171: Answer = B
Processing row 1209/1300...
✅ Processed Merge Q1171: Answer = B
Processing row 1210/1300...
⚠️ Could not extract answer from response for row 1210:
Response: To determine the most probable cause of the patient's acid-base and electrolyte disturbances, let's ...
✅ Processed Merge Q1210: Answer = None
Saved progress to CSV after 1210 items
Processing row 1211/1300...
✅ Processed Merge Q1210: Answer = D
Processing row 1212/1300...
✅ Processed Merge Q1210: Answer = A
Processing row 1213/1300...
✅ Processed Merge Q1210: Answer = D
Processing row 1214/1300...
✅ Processed Merge Q1210: Answer = A
Processing row 1215/1300...
✅ Processed Merge Q1210: Answer = D
Processing row 1216/1300...
✅ Processed Merge Q1210: Answer = B
Processing row 1217/1300...
✅ Processed Merge Q1210: Answer = A
Processing row 1218/1300...
✅ Processed Merge Q1

In [5]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "Qwen_14B_predictions_14B.csv")
df = pd.read_csv(paths.DATA / "14B_High_Low_Irr_Data_Source_.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_df3')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.5700 (57.00%)
Standard Deviation: 0.4953
95% Confidence Interval: [0.5431, 0.5969]
95% CI (percentage): [54.31%, 59.69%]
Sample Size: 1300

CATEGORY-WISE ACCURACY ANALYSIS
data_source_df3  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
           jama    582       0.644330 0.479128 0.019860     0.605323     0.683337        64.432990  47.912767      60.532281      68.333698
     medbullets    207       0.623188 0.485762 0.033763     0.556624     0.689753        62.318841  48.576182      55.662351      68.975330
       medxpert    318       0.213836 0.410659 0.023029     0.168528     0.259145        21.383648  41.065893      16.852825      25.914471
           mmlu    193       0.875648 0.330841 0.023814     0.828676     0.922619        87.564767  33.084129      82.867608      92.261926


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.5700 (57.00%)
Stan